In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import optuna
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class KingaMetricCreditRiskModel:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.feature_names = None
        self.leaky_features = ['Payment_Behaviour', 'Delay_from_due_date', 'Num_of_Delayed_Payment']
    
    def load_and_preprocess(self, filepath):
        df = pd.read_csv(filepath)
        print(f'Dataset shape: {df.shape}, Default rate: {df["Default_Flag"].mean():.3f}')
        
        # Drop leaky features (future/default indicators)
        df = df.drop(columns=[c for c in self.leaky_features if c in df.columns], errors='ignore')
        print('Dropped leaky features:', [c for c in self.leaky_features if c in df])
        
        # Outlier clipping + fillna
        df.replace([np.inf, -np.inf], np.nan, inplace=True)

        for col in df.select_dtypes('number').columns:
            Q1, Q3 = df[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            df[col] = df[col].clip(lower=Q1-1.5*IQR, upper=Q3+1.5*IQR)
        df.fillna(df.median(numeric_only=True), inplace=True)
        
        # Categorical encoding
        cat_cols = [c for c in ['Payment_of_Min_Amount', 'Credit_Mix', 'Borrower_Tier'] if c in df.columns]
        
        for col in cat_cols:
            if df[col].dtype == 'category':
                df[col] = df[col].astype(str)
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            self.label_encoders[col] = le
        
        X = df.drop('Default_Flag', axis=1)
        y = df['Default_Flag']
        self.feature_names = X.columns.tolist()
        
        return X, y
    

    def feature_selection(self, X, y):
        base_model = XGBClassifier(n_estimators=200, max_depth=4, random_state=42, enable_categorical=False)
        base_model.fit(X, y)

        importances = pd.Series(base_model.feature_importances_, index=X.columns).sort_values(ascending=False)
        top_features = importances.head(25).index  
        
        print('Top features:', top_features.tolist())

        return X[top_features], top_features
    
    def objective(self, trial, X_temp, y_temp, folds):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
            'gamma': trial.suggest_float('gamma', 0, 1),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 5),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 4),
            'random_state': 42,
            'tree_method': 'hist',
            'eval_metric': 'auc'
        }
        
        auc_scores = []

        for train_idx, val_idx in folds.split(X_temp, y_temp):
            X_train, X_val = X_temp.iloc[train_idx], X_temp.iloc[val_idx]
            y_train, y_val = y_temp.iloc[train_idx], y_temp.iloc[val_idx]
            
            # SMOTE oversample train
            smote = SMOTE(random_state=42)
            X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
            
            model = XGBClassifier(**params)
            model.fit(X_train_sm, y_train_sm)
            y_pred = model.predict_proba(X_val)[:,1]
            auc_scores.append(roc_auc_score(y_val, y_pred))
        
        return np.mean(auc_scores)
    
    def tune_hyperparams(self, X_temp, y_temp):
        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner(n_startup_trials=10))
        
        study.optimize(lambda trial: self.objective(trial, X_temp, y_temp, folds), n_trials=200)
        
        return study.best_params
    
    def train(self, filepath):
        print('Loading data...')
        X, y = self.load_and_preprocess(filepath)
        
        print('Feature selection...')
        X_selected, self.feature_names = self.feature_selection(X, y)
        
        print('Splitting...')
        X_temp, X_test, y_temp, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
        
        print('Tuning hyperparams...')
        best_params = self.tune_hyperparams(X_temp, y_temp)
        print('Best params:', best_params)
        
        # Scale
        X_temp_scaled = pd.DataFrame(self.scaler.fit_transform(X_temp), columns=self.feature_names, index=X_temp.index)
        X_test_scaled = pd.DataFrame(self.scaler.transform(X_test), columns=self.feature_names, index=X_test.index)
        
        # Train final
        self.model = XGBClassifier(**best_params, random_state=42, tree_method='hist', early_stopping_rounds=50, enable_categorical=True)
        self.model.fit(X_temp_scaled, y_temp, eval_set=[(X_test_scaled, y_test)], verbose=False)
        
        # Evaluate
        cv_folds = StratifiedKFold(5, shuffle=True, random_state=42)
        cv_aucs = []

        for train_idx, val_idx in cv_folds.split(X_temp_scaled, y_temp):
            fold_model = XGBClassifier(**best_params, random_state=42)
            fold_model.fit(self.scaler.transform(X_temp_scaled.iloc[train_idx]), y_temp.iloc[train_idx])

            pred = fold_model.predict_proba(self.scaler.transform(X_temp_scaled.iloc[val_idx]))[:,1]
            cv_aucs.append(roc_auc_score(y_temp.iloc[val_idx], pred))
            
        print(f'CV AUC: {np.mean(cv_aucs):.4f} (+/- {np.std(cv_aucs)*2:.4f})')
        
        test_auc = roc_auc_score(y_test, self.model.predict_proba(X_test_scaled)[:,1])
        
        print(f'Test AUC: {test_auc:.4f}')
        
        if test_auc >= 0.85:
            print('🎉 Target AUC 0.85 achieved!')
        else:
            print('Target not met, consider ensemble/more features.')
        
        return test_auc
    
    def save(self, path='kinga_model.pkl'):
        joblib.dump(self, path)
        
        print(f'Model saved to {path}')

    def predict_proba(self, X_new):
        X_new_scaled = pd.DataFrame(self.scaler.transform(X_new[self.feature_names]), columns=self.feature_names)
        
        return self.model.predict_proba(X_new_scaled)[:,1]
        

if __name__ == '__main__':
    model = KingaMetricCreditRiskModel()
    auc = model.train('datasets/kingametric_credit_risk.csv')
    
    model.save()

Loading data...
Dataset shape: (8744, 45), Default rate: 0.315
Dropped leaky features: []
Feature selection...


[I 2026-03-23 11:22:04,812] A new study created in memory with name: no-name-a11cecef-00ca-412d-af93-4294bd8358ef


Top features: ['Borrower_Tier', 'credit_mix_quality', 'normalized_inquiry_intensity', 'Payment_Instability', 'Payment_of_Min_Amount', 'Repayment_Stress', 'population_density_factor', 'Credit_Instability', 'Obligation_Ratio', 'Monthly_Inhand_Salary', 'Num_Credit_Inquiries', 'normalized_dti', 'Credit_Depth', 'Liquidity_Buffer', 'Credit_History_Age', 'Annual_Income', 'Num_Bank_Accounts', 'Monthly_Balance', 'Total_EMI_per_month', 'Net_Cash_Flow', 'normalized_investment_ratio', 'Outstanding_Debt', 'Credit_Mix', 'Credit_Exposure', 'Amount_invested_monthly']
Splitting...
Tuning hyperparams...


[I 2026-03-23 11:22:09,766] Trial 0 finished with value: 0.5840165278624248 and parameters: {'n_estimators': 528, 'max_depth': 6, 'learning_rate': 0.09608895451876405, 'subsample': 0.8872072105726074, 'colsample_bytree': 0.8573885068068812, 'min_child_weight': 8, 'gamma': 0.9770169808681967, 'reg_alpha': 0.5893935320649459, 'reg_lambda': 3.2551124878815942, 'scale_pos_weight': 1.2667319372431192}. Best is trial 0 with value: 0.5840165278624248.
[I 2026-03-23 11:22:12,221] Trial 1 finished with value: 0.5788876539247271 and parameters: {'n_estimators': 579, 'max_depth': 5, 'learning_rate': 0.18896240774305378, 'subsample': 0.8412601636749937, 'colsample_bytree': 0.8220689017455407, 'min_child_weight': 7, 'gamma': 0.6664872539374711, 'reg_alpha': 1.5247789131492155, 'reg_lambda': 1.934271600476357, 'scale_pos_weight': 3.628690190063965}. Best is trial 0 with value: 0.5840165278624248.
[I 2026-03-23 11:22:15,503] Trial 2 finished with value: 0.585486490451431 and parameters: {'n_estimator

Best params: {'n_estimators': 358, 'max_depth': 4, 'learning_rate': 0.016735379937086542, 'subsample': 0.7820825418467374, 'colsample_bytree': 0.8518350613546191, 'min_child_weight': 5, 'gamma': 0.8780810779992884, 'reg_alpha': 1.947633368263863, 'reg_lambda': 4.9196819167842465, 'scale_pos_weight': 1.103866122017274}
CV AUC: 0.6327 (+/- 0.0279)
Test AUC: 0.6308
Target not met, consider ensemble/more features.
Model saved to kinga_model.pkl
